# PRCP-1017 | Automobile Price Prediction
### A business-oriented machine learning study of automobile pricing

**Project type:** Regression | **Workflow:** CRISP-DM inspired | **Deliverable:** Reproducible Jupyter Notebook

---

## Executive overview

This project studies how vehicle specifications and design choices relate to automobile prices. It combines exploratory analysis with supervised regression models and translates model findings into practical considerations for product and pricing teams.

The objective is not to claim that a model discovers the one true price of a car. A model estimates prices from patterns in the supplied historical data. Its usefulness depends on data quality, representative observations, and validation on unseen vehicles.

**Important:** The project brief provides the attribute definitions and a dataset download link, but the dataset itself was not attached with the brief. The notebook therefore loads the supplied CSV/data file when available, or downloads and extracts the linked archive when internet access is available. Metrics and conclusions are generated from the actual data at runtime; no performance figures are invented.

## 1. Business problem and objectives

Automobile pricing is influenced by several interacting factors: brand positioning, engine capacity, horsepower, vehicle dimensions, weight, fuel economy, drivetrain, and body style. Understanding these relationships can help management evaluate product configurations and estimate how changes in specifications may affect market price.

**Primary objective:** predict the `price` of a vehicle using the independent variables supplied in the project dataset.

**Management questions**
- Which vehicle attributes show the strongest relationship with price?
- How much predictive accuracy can be achieved on unseen observations?
- Which model offers the best balance of accuracy, stability, and interpretability?
- What data limitations should be considered before using predictions in a business decision?

**Scope and caution:** associations in observational data are not proof of causation. The model can support scenario analysis, but it cannot establish that changing one feature will cause a particular price change.

## 2. Dataset and attribute overview

The project brief describes 26 attributes, including `symboling`, `normalized-losses`, `make`, `fuel-type`, `aspiration`, `num-of-doors`, `body-style`, `drive-wheels`, `engine-location`, dimensions, weight, engine characteristics, fuel economy, and the target `price`.

The original Automobile Imports-85 dataset commonly represents missing values with `?` and may not contain a header row. The loading section handles this format as well as a conventional CSV with column names.

**Target variable:** `price` (continuous).  
**Problem type:** supervised regression.  
**Expected data issues:** missing values, mixed numeric/categorical columns, possible outliers, and differing measurement scales.

In [ ]:
# 3. Import libraries and configure the analysis
import os, glob, zipfile, urllib.request, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 4. Load the dataset

Place the supplied project ZIP or CSV in the same folder as this notebook. If no local file is found, the notebook attempts to download the archive from the link in the project brief. If that fails, download it manually and set `DATA_PATH` below.

The code does not silently substitute a different dataset. This is important: a project should not report results from a dataset that merely resembles the assigned one.

In [ ]:
DATA_URL = "https://d3ilbtxij3aepc.cloudfront.net/projects/CDS-Capstone-Projects/PRCP-1017-AutoPricePred.zip"
DATA_PATH = None  # Example: "Automobile_data.csv" or "imports-85.data"

if DATA_PATH is None:
    candidates = []
    for pattern in ("*.csv", "*.data", "*.txt", "*.zip"):
        candidates.extend(glob.glob(pattern))
    # Prefer a project archive or likely automobile data file over unrelated files.
    preferred = [p for p in candidates if "auto" in os.path.basename(p).lower() or "price" in os.path.basename(p).lower() or "imports" in os.path.basename(p).lower()]
    local_files = preferred or candidates
    if local_files:
        DATA_PATH = local_files[0]

if DATA_PATH is None:
    archive = "PRCP-1017-AutoPricePred.zip"
    try:
        urllib.request.urlretrieve(DATA_URL, archive)
        DATA_PATH = archive
        print("Downloaded project archive.")
    except Exception as exc:
        raise FileNotFoundError(
            "Dataset not found. Download the ZIP/CSV from the dataset link in the project brief, "
            "place it beside this notebook, and set DATA_PATH to its filename. "
            f"Download attempt detail: {exc}"
        )

print("Selected file:", DATA_PATH)

In [ ]:
# Standard column names for the UCI Automobile Imports-85 data file
AUTO_COLUMNS = [
    "symboling", "normalized-losses", "make", "fuel-type", "aspiration",
    "num-of-doors", "body-style", "drive-wheels", "engine-location",
    "wheel-base", "length", "width", "height", "curb-weight", "engine-type",
    "num-of-cylinders", "engine-size", "fuel-system", "bore", "stroke",
    "compression-ratio", "horsepower", "peak-rpm", "city-mpg",
    "highway-mpg", "price"
]

def read_project_data(path):
    if path.lower().endswith(".zip"):
        with zipfile.ZipFile(path) as z:
            members = [m for m in z.namelist() if not m.endswith("/")]
            # Prefer likely data files; ignore documentation and metadata where possible.
            likely = [m for m in members if m.lower().endswith((".csv", ".data", ".txt"))]
            if not likely:
                raise ValueError(f"No CSV/DATA/TXT file found inside archive. Contents: {members}")
            member = likely[0]
            print("Reading archive member:", member)
            with z.open(member) as f:
                raw = f.read()
            from io import BytesIO
            source = BytesIO(raw)
            return parse_table(source, member)
    return parse_table(path, path)

def parse_table(source, name):
    # Try a conventional headered CSV first.
    try:
        df0 = pd.read_csv(source)
        normalized = [str(c).strip().lower().replace("_", "-") for c in df0.columns]
        if "price" in normalized:
            df0.columns = normalized
            return df0
    except Exception:
        pass
    # Re-open if source is a path; byte streams need to be reset.
    if hasattr(source, "seek"):
        source.seek(0)
    # UCI Imports-85 is comma-separated, headerless, and uses '?' for missing data.
    df = pd.read_csv(source, header=None, names=AUTO_COLUMNS, na_values=["?", "NA", "N/A", ""])
    if df.shape[1] != len(AUTO_COLUMNS):
        raise ValueError(f"Unexpected number of columns ({df.shape[1]}) in {name}. Inspect the file format.")
    return df

df = read_project_data(DATA_PATH)
df.columns = [str(c).strip().lower().replace("_", "-") for c in df.columns]
print("Shape:", df.shape)
display(df.head())

## 5. Initial data audit

Before modelling, inspect dimensions, data types, duplicate rows, missingness, and target validity. Missing values are not automatically errors; they become a problem when their pattern is ignored or when rows are discarded without justification.

In [ ]:
print("Rows, columns:", df.shape)
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nDescriptive statistics:")
display(df.describe(include="all").T)
print("\nMissing values:")
missing = df.isna().sum().sort_values(ascending=False)
display(missing[missing > 0].to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())
if "price" not in df.columns:
    raise KeyError("Target column 'price' is missing. Confirm that the correct project dataset was loaded.")

In [ ]:
# Convert columns that should be numeric. Invalid strings become missing values.
numeric_expected = [
    "symboling", "normalized-losses", "wheel-base", "length", "width", "height",
    "curb-weight", "engine-size", "bore", "stroke", "compression-ratio",
    "horsepower", "peak-rpm", "city-mpg", "highway-mpg", "price"
]
for col in numeric_expected:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Target cannot be imputed for supervised learning. Remove rows without observed price.
before = len(df)
df = df.dropna(subset=["price"]).drop_duplicates().copy()
print(f"Removed {before-len(df)} rows with missing target and/or exact duplicate rows.")
print("Usable shape:", df.shape)
display(df["price"].describe())

## 6. Exploratory data analysis

EDA is used to understand distributions, unusual observations, and relationships worth testing. Plots below are generated directly from the loaded dataset.

In [ ]:
# Target distribution
fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(df["price"], bins=25, edgecolor="white")
ax.set(title="Distribution of automobile prices", xlabel="Price", ylabel="Number of vehicles")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(df["price"].dropna(), vert=False)
ax.set(title="Price spread and potential extreme values", xlabel="Price")
plt.tight_layout()
plt.show()

In [ ]:
# Numeric correlations with price
num_df = df.select_dtypes(include=np.number)
corr = num_df.corr(numeric_only=True)
if "price" in corr:
    price_corr = corr["price"].drop("price").sort_values()
    display(price_corr.to_frame("correlation_with_price"))
    fig, ax = plt.subplots(figsize=(9, max(4, len(price_corr)*0.28)))
    price_corr.plot(kind="barh", ax=ax)
    ax.set(title="Numeric feature correlation with price", xlabel="Pearson correlation")
    plt.tight_layout()
    plt.show()

In [ ]:
# Price by selected categorical variables (when available)
categorical_candidates = ["make", "fuel-type", "aspiration", "num-of-doors", "body-style",
                          "drive-wheels", "engine-location", "engine-type", "num-of-cylinders"]
for col in categorical_candidates:
    if col in df.columns and df[col].nunique(dropna=True) <= 25:
        summary = df.groupby(col, dropna=False)["price"].agg(["count", "median", "mean"]).sort_values("median")
        print(f"\nPrice summary by {col}")
        display(summary)
        fig, ax = plt.subplots(figsize=(10, max(3, 0.35*len(summary))))
        summary["median"].plot(kind="barh", ax=ax)
        ax.set(title=f"Median price by {col}", xlabel="Median price", ylabel=col)
        plt.tight_layout()
        plt.show()

### EDA interpretation guide

Use the displayed tables and plots to write dataset-specific observations. For example, note whether the target is skewed, which numeric variables have the strongest linear association with price, and whether categories have noticeably different median prices. Correlation is a screening tool, not evidence of causation; a low linear correlation can still be useful in a non-linear model.

## 7. Data preparation and leakage control

The modelling pipeline:
- separates the target from predictors;
- identifies numeric and categorical features from the data;
- imputes numeric missing values with the median and categorical missing values with the most frequent value;
- one-hot encodes categories, ignoring previously unseen categories at prediction time;
- scales numeric variables for linear models;
- fits all transformations using training data only, avoiding information leakage from the test set.

The test set is held out until final evaluation. Model selection uses cross-validation on the training portion.

In [ ]:
X = df.drop(columns=["price"])
y = df["price"].astype(float)

# Remove columns that contain no usable variation.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
if constant_cols:
    print("Dropping constant columns:", constant_cols)
    X = X.drop(columns=constant_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = [c for c in X_train.columns if c not in numeric_features]

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
numeric_unscaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_scaled, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])
preprocessor_tree = ColumnTransformer([
    ("num", numeric_unscaled, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

## 8. Model development and comparison

A baseline model establishes a minimum reference point. Linear regression and Ridge provide interpretable linear benchmarks. Tree ensembles can capture non-linear patterns and interactions without requiring manual feature transformations.

Models are compared using the same training folds. The primary selection metric is **RMSE**, because it penalizes large prediction errors more strongly. MAE is also reported because it is easier to interpret as the average absolute price error. R² indicates the proportion of target variation explained by the model on the evaluation sample.

In [ ]:
models = {
    "Dummy baseline (median)": (preprocessor_tree, DummyRegressor(strategy="median")),
    "Linear Regression": (preprocessor_scaled, LinearRegression()),
    "Ridge Regression": (preprocessor_scaled, Ridge(alpha=10.0)),
    "Random Forest": (preprocessor_tree, RandomForestRegressor(
        n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )),
    "Extra Trees": (preprocessor_tree, ExtraTreesRegressor(
        n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )),
    "Gradient Boosting": (preprocessor_tree, GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.04, max_depth=2, loss="huber", random_state=RANDOM_STATE
    )),
    "Hist Gradient Boosting": (preprocessor_tree, HistGradientBoostingRegressor(
        max_iter=250, learning_rate=0.06, l2_regularization=1.0, random_state=RANDOM_STATE
    ))
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}

comparison = []
fitted_models = {}
for name, (prep, estimator) in models.items():
    pipe = Pipeline([("preprocess", prep), ("model", estimator)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
    comparison.append({
        "Model": name,
        "CV MAE": -scores["test_mae"].mean(),
        "CV RMSE": -scores["test_rmse"].mean(),
        "CV R2": scores["test_r2"].mean(),
        "RMSE std": scores["test_rmse"].std()
    })
    fitted_models[name] = pipe

comparison_df = pd.DataFrame(comparison).sort_values("CV RMSE").reset_index(drop=True)
display(comparison_df)

In [ ]:
# Visual comparison of cross-validation RMSE
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = comparison_df.sort_values("CV RMSE", ascending=True)
ax.barh(plot_df["Model"], plot_df["CV RMSE"])
ax.set(title="Cross-validation model comparison", xlabel="Mean CV RMSE (lower is better)")
plt.tight_layout()
plt.show()

best_name = comparison_df.iloc[0]["Model"]
best_model = fitted_models[best_name]
print("Best model selected by mean CV RMSE:", best_name)

## 9. Final hold-out evaluation

Only after choosing the model using training cross-validation do we fit it on the full training set and evaluate it once on the untouched test set. This gives a more realistic estimate of performance on unseen examples.

In [ ]:
best_model.fit(X_train, y_train)
test_pred = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = mean_squared_error(y_test, test_pred) ** 0.5
test_r2 = r2_score(y_test, test_pred)

print(f"Selected model: {best_name}")
print(f"Test MAE : {test_mae:,.2f}")
print(f"Test RMSE: {test_rmse:,.2f}")
print(f"Test R²  : {test_r2:.3f}")

In [ ]:
# Actual vs predicted and residual diagnostics
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, test_pred, alpha=0.75)
lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
ax.plot(lims, lims, linestyle="--")
ax.set(title="Actual vs predicted prices", xlabel="Actual price", ylabel="Predicted price")
plt.tight_layout()
plt.show()

residuals = y_test - test_pred
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(test_pred, residuals, alpha=0.75)
ax.axhline(0, linestyle="--")
ax.set(title="Residual plot", xlabel="Predicted price", ylabel="Actual - predicted")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(residuals, bins=20, edgecolor="white")
ax.set(title="Distribution of prediction errors", xlabel="Residual (actual - predicted)", ylabel="Count")
plt.tight_layout()
plt.show()

## 10. Understanding model drivers

Permutation importance measures how much a fitted model's score deteriorates when one original input column is shuffled. It works with the complete pipeline, so it reports importance at the original feature level rather than producing a long list of one-hot encoded dummy columns.

Importance is predictive, not causal. A feature can appear important because it acts as a proxy for another variable.

In [ ]:
perm = permutation_importance(
    best_model, X_test, y_test, scoring="neg_root_mean_squared_error",
    n_repeats=20, random_state=RANDOM_STATE, n_jobs=-1
)
importance = pd.DataFrame({
    "Feature": X_test.columns,
    "Importance (RMSE increase)": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values("Importance (RMSE increase)", ascending=False)

display(importance.head(15))
top_imp = importance.head(15).sort_values("Importance (RMSE increase)")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_imp["Feature"], top_imp["Importance (RMSE increase)"], xerr=top_imp["Std"])
ax.set(title="Top permutation feature importances", xlabel="Increase in RMSE after shuffling")
plt.tight_layout()
plt.show()

## 11. Business interpretation and production recommendation

The final recommendation must be based on the actual output produced above:

1. **Accuracy:** compare the selected model's test MAE and RMSE with the median baseline. If the improvement is small, the model may not add enough value for operational use.
2. **Error tolerance:** translate MAE into the currency unit used by the dataset. Decide with management whether this typical error is acceptable for early-stage pricing analysis.
3. **Stability:** inspect cross-validation RMSE standard deviation. A model with slightly better average accuracy but highly variable folds may be less dependable.
4. **Explainability:** use permutation importance and partial business review. For pricing decisions, managers should understand the major drivers and limitations.
5. **Deployment readiness:** monitor prediction errors on newer vehicles, validate new input ranges, retrain when market conditions change, and keep a human review step for unusual configurations.

**Production selection rule:** choose the model with the lowest cross-validation RMSE only if its hold-out performance is materially better than the baseline and the residual analysis does not reveal unacceptable systematic errors. Otherwise, treat the result as a prototype and collect better data before deployment.

## 12. Challenges encountered and mitigation

| Challenge | Why it matters | Technique used |
|---|---|---|
| Missing values represented by `?` or blanks | Many algorithms cannot accept raw missing values | Convert placeholders to missing values and impute within the pipeline |
| Mixed numeric and categorical columns | Models require consistent numeric representations | Column-wise preprocessing and one-hot encoding |
| Different feature scales | Can distort regularized linear models | Standardization for linear models |
| Skewed prices and extreme observations | Large errors may dominate model fitting and evaluation | Inspect distributions and report MAE alongside RMSE; do not delete outliers without evidence |
| Small dataset | A single split can give unstable results | Five-fold cross-validation plus a separate hold-out set |
| Non-linear relationships and interactions | A simple linear model may underfit | Compare linear and ensemble-based regressors |
| Unseen categories at prediction time | A new make or category can break encoding | `OneHotEncoder(handle_unknown="ignore")` |
| Risk of data leakage | Inflated validation scores | Fit imputation, scaling, and encoding inside pipelines |

Add any additional issues observed during your actual run, such as unexpected columns, malformed rows, or extreme residuals. Do not claim a challenge occurred unless the data or run demonstrated it.

## 13. Limitations and ethical use

- The dataset represents a particular historical market and may not reflect current prices, inflation, taxes, regional differences, or modern vehicle technology.
- Brand and specification patterns may be confounded. Predictive importance does not establish causal impact.
- A small sample can produce uncertain estimates, especially for rare makes or unusual body styles.
- The model should not be used as the sole basis for final retail pricing, financing, insurance, or customer eligibility decisions.
- Future deployment requires external validation on recent market data, monitoring, documented model versions, and a process for correcting systematic errors.

## 14. Conclusion

This project builds a reproducible workflow for analysing automobile prices and comparing regression algorithms. It begins with data validation and exploratory analysis, applies leakage-safe preprocessing, evaluates multiple models through cross-validation, and reserves a hold-out set for final assessment. The selected model and its feature drivers provide management with a structured view of pricing patterns.

The final business value depends on the measured test error, the stability of the selected model, and whether the dataset represents the market where decisions will be made. The model is therefore best treated as a decision-support tool, not an automatic pricing authority.

## 15. Submission checklist

- [ ] Run every notebook cell from top to bottom.
- [ ] Confirm that the correct assigned dataset was loaded.
- [ ] Save the notebook with all outputs visible.
- [ ] Review the generated metric values and feature-importance chart.
- [ ] Add your name, registration number, institution, and submission date to the cover section if required.
- [ ] Ensure the dataset is included separately if your institution requires it.
- [ ] Do not submit placeholder or unexecuted model results.